# CardioIA - Fase 2

Notebook final organizado com a Parte 1 e a Parte 2 do projeto.

## 1. Importação das bibliotecas
Nesta etapa importamos as bibliotecas usadas para leitura dos arquivos, normalização de texto, vetorização TF-IDF, treinamento do modelo e avaliação.

In [ ]:
import pandas as pd
import unicodedata
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 2. Parte 1 - Extração de sintomas e sugestão de diagnóstico
Aqui o sistema lê o arquivo `frases_sintomas.txt`, compara cada frase com o `mapa_conhecimento.csv` e sugere um diagnóstico com base nos sintomas encontrados.

In [ ]:
def normalizar(texto: str) -> str:
    texto = texto.lower().strip()
    return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

def carregar_mapa(caminho_csv: str):
    df = pd.read_csv(caminho_csv)
    for col in ["Sintoma1", "Sintoma2", "DoencaAssociada"]:
        if col not in df.columns:
            raise ValueError(f"Coluna obrigatória ausente: {col}")
    return df

def identificar_diagnosticos(frases_txt: str, mapa_csv: str):
    mapa = carregar_mapa(mapa_csv)
    with open(frases_txt, "r", encoding="utf-8") as f:
        frases = [linha.strip() for linha in f if linha.strip()]

    resultados = []
    for frase in frases:
        frase_norm = normalizar(frase)
        sintomas_encontrados = []
        doencas = []

        for _, row in mapa.iterrows():
            sintoma1 = normalizar(str(row["Sintoma1"]))
            sintoma2 = normalizar(str(row["Sintoma2"]))
            doenca = str(row["DoencaAssociada"])

            if sintoma1 in frase_norm:
                sintomas_encontrados.append(row["Sintoma1"])
                doencas.append(doenca)

            if sintoma2 in frase_norm:
                sintomas_encontrados.append(row["Sintoma2"])
                doencas.append(doenca)

        if doencas:
            diagnostico = max(set(doencas), key=doencas.count)
        else:
            diagnostico = "Sem correspondência no mapa"

        resultados.append({
            "frase": frase,
            "sintomas_identificados": "; ".join(sorted(set(sintomas_encontrados))) if sintomas_encontrados else "Nenhum",
            "diagnostico_sugerido": diagnostico
        })

    return pd.DataFrame(resultados)

resultado = identificar_diagnosticos("frases_sintomas.txt", "mapa_conhecimento.csv")
resultado

## 3. Salvando o resultado da Parte 1
O arquivo `resultado_diagnosticos.csv` registra a saída do sistema com a frase analisada, os sintomas identificados e o diagnóstico sugerido.

In [ ]:
resultado.to_csv("resultado_diagnosticos.csv", index=False, encoding="utf-8-sig")
print("Arquivo 'resultado_diagnosticos.csv' gerado com sucesso.")

## 4. Parte 2 - Classificador básico de texto
Nesta etapa usamos um dataset rotulado (`dataset_risco.csv`) para classificar frases médicas em `alto risco` ou `baixo risco`. O texto é transformado com TF-IDF e o modelo escolhido foi a Regressão Logística.

In [ ]:
dados = pd.read_csv("dataset_risco.csv")
dados.head()

In [ ]:
X = dados["frase"]
y = dados["situacao"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

modelo = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

modelo.fit(X_treino, y_treino)
y_pred = modelo.predict(X_teste)

print("Acurácia:", round(accuracy_score(y_teste, y_pred), 4))
print("\nMatriz de confusão:")
print(confusion_matrix(y_teste, y_pred))
print("\nRelatório de classificação:")
print(classification_report(y_teste, y_pred))

## 5. Teste com novas frases
Após o treinamento, o modelo é testado com frases novas para verificar se consegue generalizar para exemplos não vistos no treino.

In [ ]:
novas_frases = [
    "dor no peito com suor frio",
    "leve incômodo nas costas após esforço",
    "falta de ar e cansaço intenso"
]

predicoes = modelo.predict(novas_frases)

for frase, pred in zip(novas_frases, predicoes):
    print(f"Frase: {frase} -> Predição: {pred}")

## 6. Observações sobre vieses
Como a base é simulada, o modelo pode aprender padrões muito fortes e simplificados. Em um cenário real seria necessário ampliar os dados, revisar rótulos, testar mais casos ambíguos e validar o sistema com supervisão especializada.